# Agent de récolte — Jev choisit les transitions

Première expérience dans un **monde simulé**, sans connexion au serveur Minecraft.
Jev est le seul modèle utilisé. Le code définit les skills, les préconditions,
l'état observable et l'exigence d'inventaire ; Jev score les transitions proposées.
Il ne génère ni texte, ni plan, ni code.

Boucle : **observer → vérifier l'inventaire → filtrer les transitions exécutables
→ demander les probabilités à Jev → exécuter une action → observer à nouveau**.
Si l'inventaire satisfait l'objectif, la boucle s'arrête avant tout nouvel appel.

Ce monde jouet représente les arbres, les déplacements, les bûches au sol et
l'inventaire. Les actions sont atomiques. Les mobs, la physique, les ticks,
le pathfinding et leurs latences ne sont pas représentés.

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / "jev_sandbox.py").is_file():
    ROOT = ROOT / "experiments"
if not (ROOT / "jev_sandbox.py").is_file():
    raise RuntimeError("Ouvrir le notebook depuis experiments.")
if Path(sys.executable).resolve() != (ROOT / ".venv" / "Scripts" / "python.exe").resolve():
    raise RuntimeError("Choisir le kernel Minecraft experiments - Jev 0.8B.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from jev_sandbox import JevSandbox
from jev_experiments import html_table, machine_info, request_profile

# Le GPU n'est pas utilisé, même s'il est disponible.
lab = JevSandbox(device="cpu")
CPU_THREADS = 8  # Réglage le plus rapide mesuré ici parmi 1, 2, 4, 8
torch.set_num_threads(CPU_THREADS)
print(json.dumps(machine_info(), indent=2, ensure_ascii=False))
print(f"Chargement : {lab.load_seconds:.3f} s ; limite : {lab.max_length} tokens par candidat")

## Le graphe et le rôle du code

Le graphe comporte une branche principale : **chercher un arbre → approcher
→ couper une bûche → ramasser → vérifier l'inventaire**. Après une coupe, Jev peut
couper une autre bûche ou ramasser celles au sol. Une fois l'arbre épuisé, il faut
chercher le suivant. `INCLUDE_WAIT = True` propose aussi `wait` pour révéler
les blocages du modèle. Le graphe strict l'exclut ici : les actions sont atomiques
et aucun événement ne fait évoluer le monde pendant l'attente.

Les préconditions empêchent les actions impossibles : couper sans arbre à portée,
ramasser sans objet proche, ou terminer avant d'avoir assez de bûches. Elles peuvent
réduire le choix à une seule action. Avec `SKIP_SINGLE_CHOICE = True`, le code
exécute cette transition sans appeler Jev. La trace distingue `sole_legal_action`
et `jev` : on n'attribue pas les décisions forcées au modèle.

Les paramètres numériques de la tâche sont fournis par toi. Ce prototype ne
convertit pas automatiquement une consigne libre en objectif et ne découvre pas
de nouveaux tools. Une transition du graphe correspond ici à un tool existant.

In [ ]:
from wood_agent import (ACTIONS, SimulatedWorld, Tree, WoodRequirement, baseline_policy,
                        count_logs, jev_policy, run_episode, save_episode, stopping_probes)
display(HTML(html_table([{"action": key, "description": text} for key, text in ACTIONS.items()],
                       [("action", "Action"), ("description", "Description")])) )

## Objectif déterministe

`collect_new` signifie **ramasser N bûches de plus** : avec 3 bûches au départ et
une demande de 10, l'exigence est 13 dans l'inventaire. `inventory_total` signifie
**atteindre N bûches en tout**. Les planches et les bûches au sol ne comptent pas.
Les espèces de bûches acceptées sont définies dans `wood_agent.LOG_ITEMS`.

In [ ]:
QUANTITY = 10
MODE = "collect_new"  # ou "inventory_total"
INITIAL_INVENTORY = {}  # par exemple {"oak_log": 3, "oak_planks": 64}
TREES = [Tree(4, 5), Tree(12, 5), Tree(20, 5)]
MAX_STEPS = 48
NO_PROGRESS_LIMIT = 6
INCLUDE_WAIT = False  # True : Jev peut attendre indéfiniment, jusqu'à la limite
SKIP_SINGLE_CHOICE = True  # Ne pas payer une inférence pour une seule action possible
STATE_FORMAT = "json"  # ou "compact_text" pour comparer le format du même état

world = SimulatedWorld(inventory=INITIAL_INVENTORY, trees=TREES)
requirement = WoodRequirement(QUANTITY, MODE, count_logs(world.inventory))
print("Exigence :", requirement.target_total, "bûches dans l'inventaire")
print(json.dumps(world.observe(requirement), indent=2))

## Exécuter un épisode avec Jev

Relancer cette cellule remet le monde à zéro. Chaque ligne indique les choix
réellement proposés, leurs probabilités, l'action sélectionnée et l'inventaire.
`no_progress` ou `step_limit` sont des échecs observables, pas une réussite masquée.
Une grande probabilité n'est pas une garantie que l'action est pertinente.

In [ ]:
world = SimulatedWorld(inventory=INITIAL_INVENTORY, trees=TREES)
episode = run_episode(world, jev_policy(lab, STATE_FORMAT), quantity=QUANTITY, mode=MODE,
                      max_steps=MAX_STEPS, no_progress_limit=NO_PROGRESS_LIMIT,
                      include_wait=INCLUDE_WAIT, skip_single_choice=SKIP_SINGLE_CHOICE)
display(HTML(html_table(episode["trace"], [
    ("step", "Étape"), ("inventory_before", "Inventaire avant"),
    ("inventory_after", "Inventaire après"), ("candidates", "Transitions proposées"),
    ("selected", "Action"), ("decision_source", "Origine de la décision"),
    ("probabilities", "Probabilités Jev"),
    ("model_ms", "Jev ms"), ("goal_met_after", "Objectif atteint"),
])))
print(json.dumps({key: value for key, value in episode.items() if key != "trace"}, indent=2))
print("Trace enregistrée :", save_episode(episode))

## Comparer à une règle

Ce contrôle utilise les mêmes skills et le même inventaire, mais une priorité fixe :
ramasser, couper, approcher, chercher, attendre. C'est une comparaison indépendante
du modèle, pas une partie cachée de la politique Jev. Sur cette tâche simple, une
règle peut suffire et évite le coût d'inférence.

In [ ]:
baseline = run_episode(SimulatedWorld(inventory=INITIAL_INVENTORY, trees=TREES),
                       baseline_policy, QUANTITY, MODE, max_steps=MAX_STEPS,
                       include_wait=INCLUDE_WAIT, skip_single_choice=SKIP_SINGLE_CHOICE)
comparison = [{"policy": name, **{key: result[key] for key in
              ("status", "final_logs", "decisions", "model_calls", "forced_transitions", "model_total_ms")}}
              for name, result in [("Jev", episode), ("Règle fixe", baseline)]]
display(HTML(html_table(comparison, [("policy", "Politique"), ("status", "Résultat"),
    ("final_logs", "Bûches finales"), ("decisions", "Décisions"),
    ("model_calls", "Appels modèle"), ("forced_transitions", "Transitions forcées"),
    ("model_total_ms", "Temps modèle ms")])))

## Contrôle : que se passe-t-il si on autorise l'attente ?

Dans les premiers essais, Jev choisissait `wait` à répétition, même après avoir
ajouté les arbres visibles à l'observation. L'épisode strict ne prouve donc pas
qu'il sait résoudre seul la tâche. On conserve ce contrôle : les mêmes outils,
mais l'attente est proposée et le modèle doit choisir à chaque étape.

In [ ]:
idle_episode = run_episode(SimulatedWorld(inventory=INITIAL_INVENTORY, trees=TREES),
                           jev_policy(lab, STATE_FORMAT), QUANTITY, MODE,
                           include_wait=True, skip_single_choice=False,
                           max_steps=MAX_STEPS, no_progress_limit=NO_PROGRESS_LIMIT)
display(HTML(html_table(idle_episode["trace"], [("step", "Étape"), ("selected", "Choix Jev"),
    ("probabilities", "Probabilités"), ("inventory_after", "Inventaire après")])))
print(json.dumps({k: v for k, v in idle_episode.items() if k != "trace"}, indent=2))
print("Contrôle enregistré :", save_episode(idle_episode, "idle_allowed"))

## Jev sait-il quand s'arrêter ?

Test diagnostic : on lui propose toutes les actions, y compris celles dont les
préconditions ne sont pas satisfaites, et on lui demande aussi si l'objectif est
atteint. Ces réponses ne sont **jamais exécutées**. On compare aux comptes réels.
L'arrêt de l'agent reste assuré par le contrôle déterministe d'inventaire.

In [ ]:
probes = stopping_probes(lab, counts=[0, 9, 10, 11], target=10)
display(HTML(html_table(probes, [("inventory_logs", "Inventaire"),
    ("deterministic_done", "Exigence réelle satisfaite"), ("jev_selected", "Choix Jev"),
    ("jev_stop_probability", "P(stop)"), ("jev_done_probability", "P(objectif atteint)"),
    ("elapsed_ms", "Jev ms")])))

## Expériences suivantes

- Mettre 3 bûches au départ et comparer les deux modes d'objectif.
- Remplacer `TREES` par `[Tree(4, 2)]` et demander 10 bûches : l'agent doit déclarer
  l'échec plutôt que prétendre avoir terminé.
- Mettre déjà assez de bûches avec `inventory_total` : aucun appel Jev ne doit partir.
- Tester `mask_preconditions=False` dans `run_episode` : Jev voit toutes les actions,
  mais l'exécuteur refuse encore les actions impossibles et l'arrêt prématuré.
- Tester `INCLUDE_WAIT = True` et `SKIP_SINGLE_CHOICE = False` pour mesurer la
  différence entre une liberté de choix plus large et un graphe fortement contraint.
- Modifier les descriptions dans `ACTIONS` ou l'état dans `SimulatedWorld.observe`
  pour comparer qualité, taille d'entrée et latence.

La simulation n'établit pas la fiabilité dans Minecraft réel. Pour cette étape,
il faudra un adaptateur isolé qui observe le serveur et exécute ces mêmes skills.

In [ ]:
lab.close()